# Mini Jane Street Simulator — Analysis

This notebook demonstrates how to run trading simulations and analyze performance using the `mini_jane_street` framework.

## Setup

In [ ]:
import sys
sys.path.insert(0, "../src")

from decimal import Decimal

from mini_jane_street import (
    Exchange,
    MarketMaker,
    MeanReversionTrader,
    MomentumTrader,
    RandomTaker,
    SimulationConfig,
    SimulationEngine,
    Analytics,
)

## 1. Order Book Basics

### 1.1 Create an Exchange and Place Orders

In [ ]:
from mini_jane_street import OrderBook, Order, Side, OrderType

book = OrderBook()

# Place some limit orders
book.rest_order(Order(
    order_id="ask-1",
    trader_id="trader-a",
    side=Side.SELL,
    price=Decimal("100.00"),
    quantity=10,
    filled_qty=0,
    timestamp=0.0,
    order_type=OrderType.LIMIT,
))
book.rest_order(Order(
    order_id="ask-2",
    trader_id="trader-b",
    side=Side.SELL,
    price=Decimal("100.01"),
    quantity=5,
    filled_qty=0,
    timestamp=0.0,
    order_type=OrderType.LIMIT,
))
book.rest_order(Order(
    order_id="bid-1",
    trader_id="trader-a",
    side=Side.BUY,
    price=Decimal("99.00"),
    quantity=10,
    filled_qty=0,
    timestamp=0.0,
    order_type=OrderType.LIMIT,
))

print(f"Best Bid: {book.get_best_bid()}")
print(f"Best Ask: {book.get_best_ask()}")
print(f"Mid Price: {book.get_mid_price()}")
print(f"Spread: {book.get_spread()}")

### 1.2 Matching — Limit Order Hits the Book

In [ ]:
from mini_jane_street.entities import Fill

# Submit a buy limit at 100.02 — crosses the spread against ask-1 at 100.00
incoming = Order(
    order_id="buy-cross",
    trader_id="trader-c",
    side=Side.BUY,
    price=Decimal("100.02"),
    quantity=3,
    filled_qty=0,
    timestamp=1.0,
    order_type=OrderType.LIMIT,
)

fills = book.add_order(incoming)
print(f"Fills: {fills}")
print(f"Best Ask now (after partial fill): {book.get_best_ask()}")

### 1.3 Matching — Limit Order Rests (No Cross)

In [ ]:
# Submit a buy limit at 99.50 — no cross (best ask is 100.00)
incoming = Order(
    order_id="buy-rest",
    trader_id="trader-c",
    side=Side.BUY,
    price=Decimal("99.50"),
    quantity=5,
    filled_qty=0,
    timestamp=2.0,
    order_type=OrderType.LIMIT,
)

fills = book.add_order(incoming)
print(f"Fills: {fills}")
print(f"Best Bid now: {book.get_best_bid()}")

## 2. Full Simulation

### 2.1 Configure and Run

In [ ]:
config = SimulationConfig(
    initial_price=Decimal("100.00"),
    volatility=0.5,
    tick_size=Decimal("0.01"),
    tick_interval=1.0,
    num_ticks=500,
    seed=42,
)

traders = [
    RandomTaker(trader_id="random-1", action_prob=0.05, min_qty=1, max_qty=10),
    MomentumTrader(trader_id="momentum-1", momentum_threshold=0.002, window_size=10, position_size=5),
    MeanReversionTrader(trader_id="meanrev-1", reversion_threshold=0.003, window_size=20, position_size=5),
]

mm = MarketMaker(
    trader_id="mm-1",
    inventory_target=0,
    base_spread=Decimal("0.02"),
    tick_size=Decimal("0.01"),
)

engine = SimulationEngine(config, traders, market_maker=mm)
result = engine.run()

print(f"Final Mid Price: {result.final_mid}")
print(f"Total Trades: {result.num_trades}")
print(f"Simulation Ticks: {result.num_ticks}")

### 2.2 Market Maker Performance

In [ ]:
stats = mm.stats
print(f"Market Maker Stats:")
print(f"  Position: {stats.position}")
print(f"  Realized PnL: {stats.realized_pnl}")
print(f"  Inventory skew: {stats.inventory_skew}")
print(f"  Adverse selection events: {stats.adverse_selection_count}")
print(f"  Spread captured: {stats.spread_captured}")

### 2.3 Trader Performance

In [ ]:
for trader in traders:
    s = trader.stats
    realized, unrealized = trader.compute_pnl(result.final_mid)
    print(f"{trader.trader_id}:")
    print(f"  Position: {trader.position}, Avg Cost: {trader._avg_cost}")
    print(f"  Realized PnL: {realized}, Unrealized: {unrealized}")
    print(f"  Total PnL: {realized + unrealized}")
    print(f"  Buy/Sell count: {s.num_buys}/{s.num_sells}")
    print()

## 3. Analytics

### 3.1 Performance Report

In [ ]:
report = engine.analytics.compute_metrics()
print(f"Performance Report (all trades):")
print(f"  Realized PnL: {report.realized_pnl}")
print(f"  Total PnL: {report.total_pnl}")
print(f"  Sharpe Ratio: {report.sharpe_ratio}")
print(f"  Max Drawdown: {report.max_drawdown}")
print(f"  Win Rate: {report.win_rate:.2%}")
print(f"  Profit Factor: {report.profit_factor}")
print(f"  Number of Trades: {report.num_trades}")

### 3.2 Market Maker Report

In [ ]:
mm_report = engine.analytics.compute_metrics(trader_id="mm-1")
print(f"Market Maker Report:")
print(f"  Realized PnL: {mm_report.realized_pnl}")
print(f"  Win Rate: {mm_report.win_rate:.2%}")
print(f"  Sharpe: {mm_report.sharpe_ratio}")

asr = engine.analytics.compute_adverse_selection_ratio("mm-1", window=5)
print(f"  Adverse Selection Ratio (5-tick): {asr:.2%}")

### 3.3 Equity Curve

In [ ]:
equity_df = engine.analytics.compute_equity_curve()
equity_df.head(10)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Equity curve
axes[0].plot(equity_df["timestamp"], equity_df["equity"], linewidth=1.5)
axes[0].set_xlabel("Time (ticks)")
axes[0].set_ylabel("Cumulative PnL")
axes[0].set_title("Equity Curve — All Traders")
axes[0].grid(True, alpha=0.3)

# PnL per trade
axes[1].bar(range(len(equity_df)), equity_df["pnl"], alpha=0.7)
axes[1].set_xlabel("Trade #")
axes[1].set_ylabel("PnL")
axes[1].set_title("Per-Trade PnL")
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

### 3.4 Price History

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

prices = engine._market_gen.price_history
ax.plot(prices, linewidth=0.8, color="steelblue")
ax.set_xlabel("Time (ticks)")
ax.set_ylabel("Mid Price")
ax.set_title("Simulated Price Path")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Export

In [ ]:
from pathlib import Path

output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

engine.analytics.export_json(output_dir / "report.json")
engine.analytics.export_trades_csv(output_dir / "trades.csv")

print(f"Exported to {output_dir.resolve()}")

## 5. Stress Test — Many Simulations

In [ ]:
results = []
for seed in range(10):
    cfg = SimulationConfig(
        initial_price=Decimal("100.00"),
        volatility=0.5,
        tick_size=Decimal("0.01"),
        num_ticks=200,
        seed=seed,
    )
    eng = SimulationEngine(
        cfg,
        traders=[RandomTaker(trader_id="rt", action_prob=0.05)],
        market_maker=MarketMaker(trader_id="mm", base_spread=Decimal("0.02")),
    )
    res = eng.run()
    mm_pnl = eng._market_maker.stats.realized_pnl
    results.append({
        "seed": seed,
        "trades": res.num_trades,
        "mm_pnl": float(mm_pnl),
    })

import pandas as pd
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
print(f"\nMM PnL stats: mean={df_results['mm_pnl'].mean():.2f}, std={df_results['mm_pnl'].std():.2f}")